# Phase 1 — Tensors: PyTorch's One Big Idea

Everything in deep learning is a **tensor**. Your input data is a tensor. The model's
weights are tensors. The predictions, the error, the corrections sent back through the
network — all tensors. Learn this one object and the rest of PyTorch is just functions that
take tensors and give you tensors back.

### So what actually *is* a tensor?

If you've done the NumPy phases, here's the good news: **you already know 90% of this.**
A tensor is a NumPy array with two superpowers bolted on.

| | `list` (Python) | `ndarray` (NumPy) | `Tensor` (PyTorch) |
|---|---|---|---|
| Holds numbers efficiently | ✗ | ✓ | ✓ |
| Fast maths built in | ✗ | ✓ | ✓ |
| Can move onto a GPU | ✗ | ✗ | **✓** |
| Remembers how it was calculated | ✗ | ✗ | **✓** |

Those last two rows are the whole reason PyTorch exists:

1. **It can move to a GPU.** A graphics card does thousands of simple sums simultaneously.
   Training that takes a day on your CPU can take twenty minutes on a GPU. Same code — you
   just tell the tensor to move house.
2. **It remembers its own history.** A tensor keeps a record of every operation used to
   create it. That sounds like trivia, but it's what lets PyTorch work backwards and figure
   out how to improve your model. That's Phase 2, and it's the actual magic.

The word "tensor" sounds intimidating. It just means "numbers in a grid, with any number of
dimensions":

- **0 dimensions** — a single number (a *scalar*). One test score.
- **1 dimension** — a row of numbers (a *vector*). One student's marks.
- **2 dimensions** — a table (a *matrix*). A whole class's marks.
- **3 dimensions** — a stack of tables. A colour image: height × width × 3 colours.
- **4 dimensions** — a stack of those. A batch of 32 images at once.

That's it. No deeper meaning. A tensor is a box of numbers that knows its own shape.

## What this notebook covers
```
making tensors -> their properties -> number types & which chip they live on
    -> picking out values -> maths & broadcasting -> changing shape
    -> swapping with NumPy -> a taste of why we need Phase 2
```

## Section 1 — Making Tensors

Two ways to get a tensor, and you'll use both constantly:

1. **From data you already have** — `torch.tensor([1, 2, 3])`. Use this for real inputs.
2. **From a factory function** — `torch.zeros(3, 3)`, `torch.randn(2, 2)`. Use this when you
   need a tensor of the right *size* to fill in later. This is how every model's starting
   weights get created.

### Why `manual_seed` comes first

Neural networks start with **random** weights, so two runs of the same code normally give
slightly different answers. That makes it impossible to tell whether a change you made
helped, or whether you just got lucky.

`torch.manual_seed(42)` pins the randomness down: same seed, same "random" numbers, every
time. Now if the result changes, it's because *you* changed something. Set it at the top of
any notebook you want to be able to trust.

In [1]:
import torch

# Pin the randomness so this notebook gives the same numbers every run.
# Without this, every re-run shifts the results and you can't tell whether
# a change you made helped or you just got a luckier roll.
torch.manual_seed(42)

# ── 1. FROM DATA YOU ALREADY HAVE ────────────────────────────────────────────
# torch.tensor() copies your numbers into a new tensor. Same job as np.array().
# How deeply you nest the lists decides how many dimensions you get.
t_list   = torch.tensor([1, 2, 3, 4, 5])          # from a list -> 1D tensor (vector)
t_float  = torch.tensor([1.0, 2.0, 3.0])          # the .0s make this float32 -- see Section 2
t_nested = torch.tensor([[1, 2, 3], [4, 5, 6]])   # from nested list -> 2D tensor (matrix)

print('=== From Python Data ===')
print(f'1D tensor : {t_list}')
print(f'2D tensor :\n{t_nested}')

# ── 2. FROM A FACTORY FUNCTION ───────────────────────────────────────────────
# When you don't have the numbers yet and just want the right SIZE to fill in
# later. This is how every model's starting weights get made.
zeros  = torch.zeros(3, 3)          # all zeros, shape (3, 3)
ones   = torch.ones(2, 4)           # all ones, shape (2, 4)
rand   = torch.rand(5, 3)           # uniform random in [0, 1), shape (5, 3)
randn  = torch.randn(2, 2)          # bell curve around 0 -- the usual choice for fresh weights,
                                    # because a mix of small +/- values breaks symmetry
arange = torch.arange(0, 10, 2)     # like Python range(): start, stop, step
linspc = torch.linspace(0, 1, 5)    # 5 evenly spaced values between 0 and 1
eye    = torch.eye(3)               # 3x3 identity matrix

print('\n=== Factory Functions ===')
print(f'zeros(3,3)     :\n{zeros}')
print(f'ones(2,4)      :\n{ones}')
print(f'rand(5,3)      :\n{rand}')
print(f'randn(2,2)     :\n{randn}')
print(f'arange(0,10,2) : {arange}')
print(f'linspace(0,1,5): {linspc}')
print(f'eye(3)         :\n{eye}')

# ── 3. WHAT A TENSOR KNOWS ABOUT ITSELF ──────────────────────────────────────
# Get in the habit of printing shape and dtype the moment anything errors.
# Between them they explain the large majority of PyTorch problems.
x = torch.rand(4, 3)
print('\n=== Tensor Attributes ===')
print(f'shape  : {x.shape}      (rows, cols) -- same as x.size()')
print(f'ndim   : {x.ndim}       number of dimensions')
print(f'dtype  : {x.dtype}   element data type')
print(f'device : {x.device}       where the data lives (cpu/cuda/mps)')
print(f'numel  : {x.numel()}      total number of elements')

=== From Python Data ===
1D tensor : tensor([1, 2, 3, 4, 5])
2D tensor :
tensor([[1, 2, 3],
        [4, 5, 6]])

=== Factory Functions ===
zeros(3,3)     :
tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])
ones(2,4)      :
tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.]])
rand(5,3)      :
tensor([[0.8823, 0.9150, 0.3829],
        [0.9593, 0.3904, 0.6009],
        [0.2566, 0.7936, 0.9408],
        [0.1332, 0.9346, 0.5936],
        [0.8694, 0.5677, 0.7411]])
randn(2,2)     :
tensor([[-1.2284,  0.5294],
        [ 1.2211,  0.1511]])
arange(0,10,2) : tensor([0, 2, 4, 6, 8])
linspace(0,1,5): tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])
eye(3)         :
tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])

=== Tensor Attributes ===
shape  : torch.Size([4, 3])      (rows, cols) -- same as x.size()
ndim   : 2       number of dimensions
dtype  : torch.float32   element data type
device : cpu       where the data lives (cpu/cuda/mps)
numel  : 12      total nu

## Section 2 — Number Types and Which Chip They Live On

Two properties decide how a tensor behaves, and mismatches in either are probably the two
most common errors you'll hit as a beginner.

### `dtype` — what kind of number is in each slot

Everything in the tensor is the same type. The two you'll deal with:

- **`float32`** — decimals. **This is what neural networks run on.** Weights, inputs,
  predictions, gradients: all float32.
- **`int64`** — whole numbers. Used for *labels* — "this image is a 7" — and for counting.

The trap: `torch.tensor([1, 2, 3])` gives you **int64**, because you typed whole numbers.
Feed that to a network expecting float32 and it will complain. Adding one decimal point —
`[1.0, 2.0, 3.0]` — gives you float32 instead. Most "expected Float but found Long" errors
are exactly this.

### `device` — which chip the numbers physically sit on

A tensor lives either in your computer's normal memory (**`cpu`**) or in a graphics card's
own memory (**`cuda`** for NVIDIA, **`mps`** for Apple Silicon).

**Two tensors can only work together if they're on the same chip.** They can't reach across —
there's no shared wire between CPU memory and GPU memory. Mixing them gives you
`Expected all tensors to be on the same device`, and the fix is always to move one across
with `.to(device)`.

The device-detecting snippet below is worth copying into every project you write. It uses a
GPU when one exists and quietly falls back to the CPU when it doesn't, so the same notebook
runs on your laptop and on a rented server without edits.

In [2]:
# ── WHAT KIND OF NUMBER IS IN EACH SLOT ──────────────────────────────────────
# Watch the difference a decimal point makes on the next two lines:
int_t    = torch.tensor([1, 2, 3])                        # whole numbers -> int64 ('long')
float_t  = torch.tensor([1.0, 2.0, 3.0])                  # decimals -> float32, what networks use
explicit = torch.tensor([1, 2, 3], dtype=torch.float32)   # or just say what you want outright

print('=== Data Types ===')
print(f'int_t.dtype    : {int_t.dtype}')
print(f'float_t.dtype  : {float_t.dtype}')
print(f'explicit.dtype : {explicit.dtype}')

# Converting between types. You'll do this constantly, because loss functions
# want float32 and the labels you load from a dataset are usually int64.
casted  = int_t.float()               # int64 -> float32
casted2 = float_t.long()              # float32 -> int64. WARNING: chops decimals off, no rounding
casted3 = float_t.to(torch.float64)   # .to() is the general-purpose way to cast/move
print(f'\nint_t.float()       : {casted} dtype={casted.dtype}')
print(f'float_t.long()      : {casted2} dtype={casted2.dtype}')
print(f'float_t.to(float64) : {casted3} dtype={casted3.dtype}')

# ── WHICH CHIP THE NUMBERS LIVE ON ───────────────────────────────────────────
# A graphics card does thousands of small sums at the same time, so training
# on one is often 10-50x faster than on the CPU.
#
# This block is worth copying into every project: it picks the best chip
# available and falls back to the CPU, so the same notebook runs anywhere
# without you editing anything.
device = (
    'cuda' if torch.cuda.is_available()               # NVIDIA GPU
    else 'mps' if torch.backends.mps.is_available()    # Apple Silicon GPU
    else 'cpu'                                          # fallback
)
print(f'\nAvailable device: {device}')

cpu_tensor = torch.rand(2, 2)
moved = cpu_tensor.to(device)   # copies the data across. The original stays where it was --
                                # .to() gives you a NEW tensor, it doesn't move the old one
print(f'cpu_tensor.device : {cpu_tensor.device}')
print(f'moved.device      : {moved.device}')

# ⚠️  TENSORS ON DIFFERENT CHIPS CANNOT WORK TOGETHER.
# There's no wire between CPU memory and GPU memory, so cpu_tensor + moved
# fails with 'Expected all tensors to be on the same device' whenever a GPU
# is in play. The fix is always the same: .to(device) one of them first.

=== Data Types ===
int_t.dtype    : torch.int64
float_t.dtype  : torch.float32
explicit.dtype : torch.float32

int_t.float()       : tensor([1., 2., 3.]) dtype=torch.float32
float_t.long()      : tensor([1, 2, 3]) dtype=torch.int64
float_t.to(float64) : tensor([1., 2., 3.], dtype=torch.float64) dtype=torch.float64

Available device: mps
cpu_tensor.device : cpu
moved.device      : mps:0


## Section 3 — Picking Out Values

Identical to NumPy — same brackets, same colons, same rules. If you did Phase 2 of the NumPy
notebooks, skim this and move on.

The quick recap:
- `t[0]` — one row
- `t[0, 2]` — one specific cell (row 0, column 2)
- `t[:, 0]` — a colon means "all of them", so this is the whole first column
- `t[t > 6]` — a condition, keeping only the values that pass
- `t[[0, 2]]` — a list of exactly which rows you want

### The one genuinely new thing: `.item()`

Pull a single value out of a tensor and you don't get a plain number — you get a *tensor
that happens to contain one number*. It still carries all the tensor machinery around with
it.

`.item()` strips that away and hands you an ordinary Python number.

You'll use it constantly when printing progress during training: `loss.item()` gives you
`0.4231` instead of `tensor(0.4231, grad_fn=<MseLossBackward0>)`. It also matters for
memory — holding on to a raw loss tensor keeps its whole calculation history alive, and
across thousands of training steps that adds up.

In [3]:
# The numbers 1 to 12, arranged as 3 rows of 4 -- something to practise on
t = torch.arange(1, 13).reshape(3, 4)   # [[1,2,3,4], [5,6,7,8], [9,10,11,12]]
print('=== Base Tensor ===')
print(t)

# ── BASIC INDEXING ────────────────────────────────────────────────────────────
print('\n=== Indexing ===')
print(f't[0]    = {t[0]}          -> first row')
print(f't[0, 2] = {t[0, 2].item()}                -> row 0, col 2 (scalar)')
print(f't[-1]   = {t[-1]}       -> last row')

# ── SLICING ────────────────────────────────────────────────────────────────────
print('\n=== Slicing ===')
print(f't[:, 0]     = {t[:, 0]}       -> all rows, column 0')
print(f't[0:2, 1:3] =\n{t[0:2, 1:3]}   -> rows 0-1, cols 1-2')
print(f't[:, -1]    = {t[:, -1]}       -> last column')

# ── FILTER BY A CONDITION ────────────────────────────────────────────────────
# The comparison gives you a True/False tensor the same shape as the original.
# Use it as an index and only the True positions come back.
mask = t > 6
print('\n=== Boolean Masking ===')
print(f'mask (t > 6):\n{mask}')
print(f't[t > 6]     : {t[mask]}   -> flat tensor of matching elements')

# ── ASK FOR EXACTLY THE ROWS/COLUMNS YOU WANT ────────────────────────────────
# Hand over a list of positions. Any order you like, repeats allowed.
print('\n=== Fancy Indexing ===')
print(f't[[0, 2]]     : rows 0 and 2 ->\n{t[[0, 2]]}')
print(f't[:, [0, 3]]  : cols 0 and 3 ->\n{t[:, [0, 3]]}')

# ── .item() — UNWRAP A SINGLE VALUE ──────────────────────────────────────────
# Indexing one value still gives you a TENSOR containing one number, with all
# the tensor machinery attached. .item() hands you a plain Python number.
# You'll use this constantly for printing: loss.item() prints 0.4231 rather
# than tensor(0.4231, grad_fn=<MseLossBackward0>).
scalar = t[1, 1]
print(f'\nt[1,1] as tensor : {scalar}')
print(f't[1,1].item()    : {scalar.item()}  (type: {type(scalar.item()).__name__})')

=== Base Tensor ===
tensor([[ 1,  2,  3,  4],
        [ 5,  6,  7,  8],
        [ 9, 10, 11, 12]])

=== Indexing ===
t[0]    = tensor([1, 2, 3, 4])          -> first row
t[0, 2] = 3                -> row 0, col 2 (scalar)
t[-1]   = tensor([ 9, 10, 11, 12])       -> last row

=== Slicing ===
t[:, 0]     = tensor([1, 5, 9])       -> all rows, column 0
t[0:2, 1:3] =
tensor([[2, 3],
        [6, 7]])   -> rows 0-1, cols 1-2
t[:, -1]    = tensor([ 4,  8, 12])       -> last column

=== Boolean Masking ===
mask (t > 6):
tensor([[False, False, False, False],
        [False, False,  True,  True],
        [ True,  True,  True,  True]])
t[t > 6]     : tensor([ 7,  8,  9, 10, 11, 12])   -> flat tensor of matching elements

=== Fancy Indexing ===
t[[0, 2]]     : rows 0 and 2 ->
tensor([[ 1,  2,  3,  4],
        [ 9, 10, 11, 12]])
t[:, [0, 3]]  : cols 0 and 3 ->
tensor([[ 1,  4],
        [ 5,  8],
        [ 9, 12]])

t[1,1] as tensor : 6
t[1,1].item()    : 6  (type: int)


## Section 4 — Maths and Broadcasting

### `*` and `@` are not the same thing

This is the single most important line in this notebook:

- **`a * b` multiplies item by item.** First slot times first slot, and so on. Shape stays
  the same.
- **`a @ b` is matrix multiplication** — rows against columns, summed. Shape changes.

Both are valid Python on the same two tensors, so **getting it wrong produces no error** —
just a wrong answer that trains into a model that doesn't work. Every neural network layer
is a matrix multiplication, so when you mean that, write `@`.

### Broadcasting — different shapes, no loop

Add a tensor of 3 numbers to a 3×3 table and PyTorch doesn't complain — it applies those 3
numbers to *every row*. That's **broadcasting**: the smaller tensor is stretched to fit.

The rule (identical to NumPy): line the shapes up from the right-hand end. Each pair must
either match, or have a 1 in one of them (a 1 stretches).

Nothing is actually copied — PyTorch just re-reads the same values. This is how a bias gets
added to every row in a batch at once.

### Reductions and `dim`

`sum`, `mean`, `max` squash a tensor down. **PyTorch calls the direction `dim`; NumPy calls
it `axis`. Same idea, different word** — and it still means *the direction that disappears*.

- `dim=0` squashes the rows away → one answer per column
- `dim=1` squashes the columns away → one answer per row

### Operations ending in `_`

A trailing underscore means **in-place**: `c.add_(10)` changes `c` itself rather than
handing back a new tensor. It saves memory, which is why it exists.

⚠️ Use these sparingly. In-place edits can destroy the history PyTorch needs for learning,
producing `a variable needed for gradient computation has been modified`. Phase 2 explains
why. Until then: prefer `c = c + 10`.

In [4]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

# ── ITEM-BY-ITEM ARITHMETIC ──────────────────────────────────────────────────
# First slot with first slot, second with second, and so on.
print('=== Element-wise Ops ===')
print(f'a + b = {a + b}')
print(f'a - b = {a - b}')
print(f'a * b = {a * b}   <- NOT matrix multiplication, just element-wise')
print(f'a / b = {a / b}')

# ── BROADCASTING — DIFFERENT SHAPES, NO LOOP ─────────────────────────────────
# The smaller tensor gets reused against the bigger one. Nothing is actually
# copied -- PyTorch just re-reads the same values, so it costs no extra memory.
#
# The rule: line the shapes up from the RIGHT. Each pair must either match, or
# have a 1 in one of them (the 1 stretches).
#
# This is how a bias gets added to every row of a batch in one go.
m   = torch.ones(3, 3)
row = torch.tensor([1.0, 2.0, 3.0])   # shape (3,)
print('\n=== Broadcasting: (3,3) + (3,) ===')
print(f'm:\n{m}')
print(f'row: {row}')
print(f'm + row:\n{m + row}   <- row is added to every row of m')

# ── MATRIX MULTIPLICATION — THE ONE TO GET RIGHT ─────────────────────────────
# @ is rows-against-columns, summed. * is item by item. They're both legal on
# these two tensors, so picking the wrong one gives NO error -- just a wrong
# answer. Every neural network layer is a matrix multiplication, so use @.
A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
B = torch.tensor([[5.0, 6.0], [7.0, 8.0]])
print('\n=== Matrix Multiplication ===')
print(f'A @ B (same as torch.matmul):\n{A @ B}')
print(f'A * B (element-wise, NOT matmul):\n{A * B}')

# ── SQUASHING A TENSOR DOWN TO FEWER NUMBERS ─────────────────────────────────
# 'dim' is PyTorch's word for NumPy's 'axis', and it means the same thing:
# THE DIRECTION THAT DISAPPEARS. dim=0 squashes the rows away (leaving one
# answer per column); dim=1 squashes the columns away (one answer per row).
x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print('\n=== Reductions ===')
print(f'x.sum()      = {x.sum()}          total')
print(f'x.sum(dim=0) = {x.sum(dim=0)}    column-wise sum (collapses rows)')
print(f'x.sum(dim=1) = {x.sum(dim=1)}    row-wise sum (collapses columns)')
print(f'x.mean()     = {x.mean()}')
print(f'x.max()      = {x.max()}')
print(f'x.argmax()   = {x.argmax()}       index of max in the FLATTENED tensor')

# ── OPERATIONS ENDING IN AN UNDERSCORE ───────────────────────────────────────
# A trailing _ means 'change this tensor itself' rather than 'give me a new
# one'. c.add_(10) edits c directly. It saves memory, which is why it exists.
#
# ⚠️  Use these sparingly. Editing a tensor in place can wipe out the history
# PyTorch needs in order to learn, giving you 'a variable needed for gradient
# computation has been modified'. Phase 2 explains why. Until then, prefer
# the ordinary c = c + 10.
c = torch.tensor([1.0, 2.0, 3.0])
print(f'\nBefore add_: {c}')
c.add_(10)   # c = c + 10, done in-place
print(f'After c.add_(10): {c}')

=== Element-wise Ops ===
a + b = tensor([5., 7., 9.])
a - b = tensor([-3., -3., -3.])
a * b = tensor([ 4., 10., 18.])   <- NOT matrix multiplication, just element-wise
a / b = tensor([0.2500, 0.4000, 0.5000])

=== Broadcasting: (3,3) + (3,) ===
m:
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
row: tensor([1., 2., 3.])
m + row:
tensor([[2., 3., 4.],
        [2., 3., 4.],
        [2., 3., 4.]])   <- row is added to every row of m

=== Matrix Multiplication ===
A @ B (same as torch.matmul):
tensor([[19., 22.],
        [43., 50.]])
A * B (element-wise, NOT matmul):
tensor([[ 5., 12.],
        [21., 32.]])

=== Reductions ===
x.sum()      = 21.0          total
x.sum(dim=0) = tensor([5., 7., 9.])    column-wise sum (collapses rows)
x.sum(dim=1) = tensor([ 6., 15.])    row-wise sum (collapses columns)
x.mean()     = 3.5
x.max()      = 6.0
x.argmax()   = 5       index of max in the FLATTENED tensor

Before add_: tensor([1., 2., 3.])
After c.add_(10): tensor([11., 12., 13.]

## Section 5 — Changing Shape

Reshaping doesn't move a single number. It only changes how PyTorch *reads* the same block
of memory — 12 numbers in a row, re-read as 3 rows of 4.

This sounds like a minor detail and it is absolutely not. **Most of the errors you'll hit
building models are shape errors.** Layers demand exact input shapes, and a huge part of the
job is reshaping tensors to fit — for example an image arriving as
`(batch, channels, height, width)` needing to be flattened before a normal layer can take it.

The tools:

| Call | What it does | When you need it |
|---|---|---|
| `.view(3, 4)` / `.reshape(3, 4)` | Re-reads the same data with a new shape | Constantly |
| `.transpose(0, 1)` | Swaps **two** dimensions | Rows↔columns |
| `.permute(2, 0, 1)` | Reorders **all** dimensions at once | Image format conversions |
| `.squeeze()` | Removes dimensions of size 1 | Cleaning up after a reduction |
| `.unsqueeze(0)` | **Adds** a dimension of size 1 | Feeding one item to a model that expects a batch |
| `.flatten(start_dim=1)` | Collapses dimensions into one | Between the conv part and the dense part of a CNN |

### `view` vs `reshape` — which to use

`.view()` refuses to copy. If the data isn't laid out neatly enough in memory, it errors.
`.reshape()` tries `view` first and quietly makes a copy if it has to.

**Use `.reshape()` unless you specifically need the guarantee of no copy.** It just works.

### `unsqueeze(0)` is the one you'll reach for most

Models are built to process a *batch* of items. Show one image to a model expecting
`(batch, channels, height, width)` and your `(channels, height, width)` tensor is one
dimension short. `unsqueeze(0)` adds an empty batch dimension at the front, turning your
single image into "a batch containing exactly one image". Everything then works.

In [5]:
t = torch.arange(1, 13)   # one flat row of 12 numbers to reshape
print(f'Original: {t}  shape={t.shape}')

# ── VIEW vs RESHAPE ──────────────────────────────────────────────────────────
# Neither one moves any numbers. They just change how the same block of
# memory is READ -- 12 numbers in a row, re-read as 3 rows of 4.
#
# .view()    refuses to copy. If the data isn't laid out tidily enough, it errors.
# .reshape() tries view() first, and quietly copies if it has to.
#
# Use .reshape() unless you specifically need the no-copy guarantee. It just works.
v = t.view(3, 4)
r = t.reshape(3, 4)
print(f'\nview(3,4):\n{v}')
print(f'reshape(3,4):\n{r}')

# Because nothing was copied, all three names point at the same numbers.
# Change one and you've changed them all:
v[0, 0] = 999
print(f'\nAfter v[0,0]=999, original t: {t}   <- t changed too! (shared memory)')
t[0] = 1  # restore for later cells

# ── SWAPPING DIMENSIONS AROUND ───────────────────────────────────────────────
# transpose swaps exactly TWO dimensions. permute reorders ALL of them at once,
# which is what you need when converting between image formats (PyTorch wants
# channels first; most image libraries put channels last).
mat = torch.arange(1, 7).reshape(2, 3)
print(f'\nmat (2,3):\n{mat}')
print(f'mat.transpose(0,1) (3,2):\n{mat.transpose(0, 1)}   <- swaps dims 0 and 1')

cube = torch.rand(2, 3, 4)            # shape (2, 3, 4)
permuted = cube.permute(2, 0, 1)      # reorder dims -> shape (4, 2, 3)
print(f'\ncube.shape: {cube.shape}')
print(f'cube.permute(2,0,1).shape: {permuted.shape}   <- reorders ALL dims at once')

# ── ADDING AND REMOVING EMPTY DIMENSIONS ─────────────────────────────────────
# A dimension of size 1 holds no extra numbers, but it changes the SHAPE, and
# shapes are what layers check.
#
# unsqueeze(0) is the one you'll reach for most. Models expect a BATCH of
# items, so showing them a single image needs an empty batch dimension on the
# front: (3, 28, 28) becomes (1, 3, 28, 28), i.e. 'a batch containing one image'.
s = torch.rand(3, 1, 4)
print(f'\ns.shape: {s.shape}')
print(f's.squeeze().shape  : {s.squeeze().shape}      <- removes ALL size-1 dims')
print(f's.squeeze(1).shape : {s.squeeze(1).shape}      <- removes only dim 1')

single = torch.rand(4)
print(f'\nsingle.shape: {single.shape}')
print(f'single.unsqueeze(0).shape: {single.unsqueeze(0).shape}  <- adds a batch dim at the front')

# ── FLATTEN — SQUASH DIMENSIONS TOGETHER ─────────────────────────────────────
# start_dim=1 is the important version: it keeps the batch dimension intact
# and flattens everything after it. That's exactly the step between the
# convolution part of a CNN and the ordinary layers that follow (Phase 4).
img = torch.rand(2, 3, 4)   # e.g. (channels, height, width)
print(f'\nimg.shape: {img.shape}')
print(f'img.flatten().shape             : {img.flatten().shape}   <- collapses to 1D')
print(f'img.flatten(start_dim=1).shape  : {img.flatten(start_dim=1).shape}  <- keeps dim 0, flattens the rest')

Original: tensor([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12])  shape=torch.Size([12])

view(3,4):
tensor([[ 1,  2,  3,  4],
        [ 5,  6,  7,  8],
        [ 9, 10, 11, 12]])
reshape(3,4):
tensor([[ 1,  2,  3,  4],
        [ 5,  6,  7,  8],
        [ 9, 10, 11, 12]])

After v[0,0]=999, original t: tensor([999,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12])   <- t changed too! (shared memory)

mat (2,3):
tensor([[1, 2, 3],
        [4, 5, 6]])
mat.transpose(0,1) (3,2):
tensor([[1, 4],
        [2, 5],
        [3, 6]])   <- swaps dims 0 and 1

cube.shape: torch.Size([2, 3, 4])
cube.permute(2,0,1).shape: torch.Size([4, 2, 3])   <- reorders ALL dims at once

s.shape: torch.Size([3, 1, 4])
s.squeeze().shape  : torch.Size([3, 4])      <- removes ALL size-1 dims
s.squeeze(1).shape : torch.Size([3, 4])      <- removes only dim 1

single.shape: torch.Size([4])
single.unsqueeze(0).shape: torch.Size([1, 4])  <- adds a batch dim at the front

img.shape: torch.Size([2, 3, 4])
img.flat

## Section 6 — Swapping With NumPy

PyTorch and NumPy convert between each other for free — because **they hand over the same
memory rather than copying it**.

Free is good. But it means what you probably don't expect:

> Change the NumPy array, and the tensor changes too. They aren't two objects holding equal
> values — they're two labels on one set of numbers.

This is the same view-vs-copy idea from NumPy Phase 1, now stretching across two libraries.
It's efficient and it's a genuine source of "but I never touched that tensor!" bugs.

**When you want them truly separate, use `.clone()`** (PyTorch's word for `.copy()`).

One more rule: **`.numpy()` only works on CPU tensors.** NumPy has no idea a GPU exists, so
a tensor living on `cuda` or `mps` must come home first: `gpu_tensor.cpu().numpy()`.

In [6]:
import numpy as np

# ── NUMPY -> TORCH ───────────────────────────────────────────────────────────
# No copying happens here. Both names end up pointing at the SAME numbers.
arr = np.array([1.0, 2.0, 3.0])
t_from_np = torch.from_numpy(arr)   # shares memory with arr -- these are not two copies
print(f'numpy array : {arr}')
print(f'torch tensor: {t_from_np}')

arr[0] = 999.0   # change the NumPy array and watch the tensor change with it...
print(f'\nAfter arr[0]=999: torch tensor is {t_from_np}   <- changed too! (shared memory)')

# ── TORCH -> NUMPY ────────────────────────────────────────────────────────────
t2 = torch.tensor([4.0, 5.0, 6.0])
np_from_t = t2.numpy()   # also shares memory (CPU tensors only)
print(f'\ntensor: {t2}')
print(f'.numpy(): {np_from_t}   type={type(np_from_t).__name__}')

# ⚠️  .numpy() only works on CPU tensors. NumPy has no idea a GPU exists, so a
# tensor living on cuda/mps has to come home first: gpu_tensor.cpu().numpy()

# ── WHEN YOU WANT THEM GENUINELY SEPARATE ────────────────────────────────────
# .clone() is PyTorch's .copy(): it makes a real, independent duplicate.
t_copy = torch.from_numpy(arr).clone()
arr[0] = -1.0
print(f'\nAfter modifying arr again, t_copy (cloned) is unaffected: {t_copy}')

numpy array : [1. 2. 3.]
torch tensor: tensor([1., 2., 3.], dtype=torch.float64)

After arr[0]=999: torch tensor is tensor([999.,   2.,   3.], dtype=torch.float64)   <- changed too! (shared memory)

tensor: tensor([4., 5., 6.])
.numpy(): [4. 5. 6.]   type=ndarray

After modifying arr again, t_copy (cloned) is unaffected: tensor([999.,   2.,   3.], dtype=torch.float64)


## Section 7 — A Taste of Why Phase 2 Exists

Here's a model. Genuinely — this is a real, if tiny, one:

```
prediction = w * x + b
```

`x` is the input, `w` and `b` are the two numbers the model has to learn. Training means
finding good values for `w` and `b`.

Below we set up inputs, the correct answers, and a deliberately bad first guess (`w=0`,
`b=0`). Then we measure how wrong we are with **MSE** — mean squared error:

1. Subtract each prediction from the true answer → the error on that item
2. Square it → makes negatives positive, and punishes big misses much harder than small ones
3. Take the average → one number summarising how wrong the whole model is

One number, and lower is better. Training is just the hunt for the `w` and `b` that make it
as small as possible.

### And here's exactly where we get stuck

We know **how wrong** we are. We do not know **which way to move**.

Should `w` go up or down? By how much? With two parameters you could guess and check. Real
networks have millions, and guessing is hopeless.

Working it out properly means calculus — the derivative of the loss with respect to every
parameter, chain rule and all. Doing that by hand for a million parameters is not happening.

**That's the problem autograd solves.** PyTorch tracks every operation you perform, then
works backwards through them and computes all million derivatives automatically. One method
call: `loss.backward()`.

That's Phase 2, and it's the heart of the whole framework.

In [7]:
# A real (if tiny) model: prediction = w * x + b.
# x is the input, y is the right answer, and w and b are the two numbers the
# model has to learn. Training = finding good values for w and b.
x = torch.tensor([1.0, 2.0, 3.0, 4.0])   # inputs
y = torch.tensor([3.0, 5.0, 7.0, 9.0])   # true outputs (the real relationship is y = 2x + 1)

# A wrong first guess for the parameters
w = torch.tensor(0.0)    # should end up near 2.0
b = torch.tensor(0.0)    # should end up near 1.0

# ── FORWARD PASS — RUN THE DATA THROUGH AND SEE WHAT COMES OUT ───────────────
# 'Forward pass' just means: make a prediction. That's all the phrase means.
y_pred = w * x + b

# ── LOSS — ONE NUMBER SAYING HOW WRONG WE ARE ────────────────────────────────
# Mean Squared Error, read right to left:
#   (y_pred - y)  -> how far off each prediction is
#   ** 2          -> square it: makes negatives positive, and punishes big
#                    misses far more harshly than small ones
#   .mean()       -> average them into a single score
# Lower is better. Training is the hunt for the w and b that make this smallest.
loss = ((y_pred - y) ** 2).mean()

print(f'inputs (x)         : {x}')
print(f'true values (y)    : {y}')
print(f'predicted (y_pred) : {y_pred}')
print(f'loss (MSE)         : {loss.item():.4f}')

# ── AND HERE'S WHERE WE'RE STUCK ─────────────────────────────────────────────
# We know HOW WRONG we are. We don't know WHICH WAY TO MOVE.
#
# Should w go up or down? By how much? With two parameters you could guess and
# check. Real networks have millions of them, and guessing is hopeless.
#
# Doing it properly means calculus -- the derivative of the loss with respect
# to every single parameter. By hand, for a million parameters, that is simply
# not happening.
#
# That is exactly the problem autograd solves. PyTorch records every operation
# you perform, then walks backwards through them and works out all million
# derivatives for you. One method call: loss.backward().
#
# That's Phase 2, and it's the heart of the entire framework.

inputs (x)         : tensor([1., 2., 3., 4.])
true values (y)    : tensor([3., 5., 7., 9.])
predicted (y_pred) : tensor([0., 0., 0., 0.])
loss (MSE)         : 41.0000


## Phase 1 Summary

### Cheat sheet
```
Making tensors:
  torch.tensor([...])         -> from data you already have
  torch.zeros/ones/rand/randn(shape)   -> a blank of the right size
  torch.arange(start,stop,step)  /  torch.linspace(start,stop,count)
  torch.eye(n)                -> identity matrix
  torch.manual_seed(42)       -> make the randomness repeatable

Checking what you've got  (print these two when anything breaks):
  t.shape   t.dtype           <- these explain most errors
  t.ndim    t.device    t.numel()

Types and devices:
  t.float() / t.long() / t.to(torch.float64)
  t.to(device)                -> move between cpu / cuda / mps
  the pattern worth memorising:
    device = "cuda" if torch.cuda.is_available() \
             else "mps" if torch.backends.mps.is_available() else "cpu"

Picking values:
  t[i]  t[i, j]  t[:, j]  t[i:j, k:l]
  t[t > 5]                    -> filter by condition
  t[[0, 2]]                   -> exactly these rows
  t.item()                    -> one-value tensor -> plain Python number

Maths:
  + - * /                     -> item by item (broadcasting applies)
  a @ b                       -> MATRIX multiplication (not the same as *)
  t.sum(dim=), t.mean(), t.max(), t.argmax()
  t.add_(x)                   -> in-place (the trailing underscore)

Changing shape:
  t.reshape(shape)            -> your default
  t.view(shape)               -> same, but refuses to copy
  t.transpose(a, b)  /  t.permute(*dims)
  t.squeeze()  /  t.unsqueeze(dim)      -> remove / add a size-1 dimension
  t.flatten(start_dim=)

NumPy bridge:
  torch.from_numpy(arr)  /  t.numpy()   -> SHARED memory (CPU only)
  .clone()                              -> break the link
```

### The five things worth memorising

1. **A tensor is a NumPy array that can use a GPU and remembers its own history.** Those two
   extras are the whole point of PyTorch.
2. **When something breaks, print `.shape` and `.dtype` first.** Shape mismatches and
   int-vs-float mistakes cause most beginner errors.
3. **`*` is item by item; `@` is matrix multiplication.** Nothing will warn you if you pick
   the wrong one.
4. **Two tensors must be on the same device to work together.**
5. **`from_numpy` and `.numpy()` share memory.** Use `.clone()` when you want independence.

### Test yourself before moving on

1. Make a tensor of shape `(2, 3, 4)` filled with random numbers and print its `shape`,
   `ndim` and `numel`. Can you predict `numel` before running it?
2. You have one image of shape `(3, 28, 28)` and a model that expects
   `(batch, 3, 28, 28)`. Which single call fixes it?
3. `a` is `(2, 3)` and `b` is `(3, 2)`. Which of `a * b` and `a @ b` works? What shape comes
   out?
4. Make a tensor from a NumPy array, change the array, and print the tensor. Explain the
   result to yourself before you look.

### Up next — Phase 2: Autograd
`requires_grad=True`, `loss.backward()`, `t.grad`, `torch.no_grad()` — how PyTorch works out
which way to nudge every parameter.